<a href="https://colab.research.google.com/github/elias9080dm/XenoTox/blob/main/Ligand_Curation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Ligand Curation**

## Import libraries and load files

In [19]:
!pip install rdkit

In [20]:
target = 'ahr' # 'ahr', 'car', 'pxr'
confidence = 'hc' # 'hc', 'lc --> high / low

In [21]:
# Montar drive
from google.colab import drive
import os, sys
from pathlib import Path

drive.mount('/content/drive')

# Crear carpetas
BASE_DIR = "/content/drive/MyDrive/QSAR/xenotox"
os.makedirs(f"{BASE_DIR}/ligands/{target}", exist_ok=True)

# Establecer carpeta padre
PROJECT_ROOT = Path("/content/drive/MyDrive/QSAR/xenotox")
sys.path.append(str(PROJECT_ROOT))



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
import pandas as pd
outpath = f"{BASE_DIR}/ligands/{target}/datatable_all_{target}.csv"
csv = pd.read_csv(outpath)
selected_columns = ['CID', 'SMILES', 'Agonist_Activity', 'Agonist_Potency (uM)', 'Agonist_Efficacy (%)', 'Viability_Activity', 'Viability_Potency (uM)', 'Viability_Efficacy (%)']
df = csv[selected_columns]
df['Agonist_Activity'] = df['Agonist_Activity'].replace('active agonist', 'active')
df.info()
df.head(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10486 entries, 0 to 10485
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   CID                     10337 non-null  float64
 1   SMILES                  10337 non-null  object 
 2   Agonist_Activity        10486 non-null  object 
 3   Agonist_Potency (uM)    1530 non-null   float64
 4   Agonist_Efficacy (%)    9729 non-null   float64
 5   Viability_Activity      10486 non-null  object 
 6   Viability_Potency (uM)  1188 non-null   float64
 7   Viability_Efficacy (%)  10294 non-null  float64
dtypes: float64(5), object(3)
memory usage: 655.5+ KB


/tmp/ipykernel_8957/1520459595.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Agonist_Activity'] = df['Agonist_Activity'].replace('active agonist', 'active')


,CID,SMILES,Agonist_Activity,Agonist_Potency (uM),Agonist_Efficacy (%),Viability_Activity,Viability_Potency (uM),Viability_Efficacy (%)
0,12850184.0,C(C(=O)[C@H]([C@@H]([C@H](C(=O)[O-])O)O)O)O.C(...,inactive,NaN,0.0000,inactive,NaN,0.0
1,89753.0,C([C@H]([C@H]([C@@H]([C@H](C(=O)[O-])O)O)O)O)O...,inactive,NaN,0.0000,inactive,NaN,0.0
2,9403.0,C[C@]12CC[C@H]3[C@H]([C@@H]1CC[C@@H]2OC(=O)CCC...,inactive,NaN,0.0000,inactive,NaN,0.0
3,13218779.0,C[C@@]12CC[C@@H](C1(C)C)C[C@H]2OC(=O)CSC#N,inactive,NaN,0.0000,inactive,NaN,0.0
4,142766.0,C1=CC=C(C(=C1)C(=O)O)O.C1=CC2=C(C(=C1)O)N=CC=C2,inactive,NaN,0.0000,inconclusive antagonist,NaN,NaN
5,16043.0,CC(C)(C)C1=C(C=CC(=C1)O)O,active,17.7801,72.1656,inactive,NaN,0.0
6,16043.0,CC(C)(C)C1=C(C=CC(=C1)O)O,active,16.7855,153.3980,inactive,NaN,0.0
7,2724411.0,CN(C)C1=CC=C(C=C1)C(=C2C=CC(=[N+](C)C)C=C2)C3=...,active,10.5909,62.5957,inconclusive antagonist,NaN,NaN
8,90454.0,C=C[C@H]1CN2CC[C@H]1C[C@@H]2[C@H](C3=CC=NC4=CC...,inconclusive agonist,NaN,NaN,inactive,NaN,0.0
9,637566.0,CC(=CCC/C(=C/CO)/C)C,inactive,NaN,0.0000,inactive,NaN,0.0


## Curation


In [23]:
from rdkit import Chem
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit import RDLogger
import pandas as pd

RDLogger.DisableLog("rdApp.*")

SMILES_COL = "SMILES"

# =============================================================================
# INITIALIZATION
# =============================================================================
df_qsar = df.copy()
total_initial = len(df_qsar)
df_qsar["SMILES_raw"] = df_qsar[SMILES_COL]

curation_report = []
std_report = []

def add_report(step, desc, before, after):
    rem = before - after
    pct = round(rem / before * 100, 2) if before else 0
    curation_report.append({
        "Step": step,
        "Description": desc,
        "Entries_before": before,
        "Entries_after": after,
        "Removed": rem,
        "Removed_%": pct
    })

# =============================================================================
# STEP 1 – SEMANTIC FILTERING
# =============================================================================

# LOW CONFIDENCE
if confidence == 'low':
    before = len(df_qsar)

    mask = (
        df_qsar["Agonist_Activity"].str.lower().isin(["active", "inactive"])
    )

    df_qsar = df_qsar[mask].reset_index(drop=True)

    add_report(1, "Semantic filtering", before, len(df_qsar))

    print("Actives:", (df_qsar["Agonist_Activity"] == "active").sum())
    print("Inactives:", (df_qsar["Agonist_Activity"] == "inactive").sum())

# HIGH CONFIDENCE
else:
    before = len(df_qsar)

    # --- HIGH CONFIDENCE ACTIVE ---
    mask_active = (
    (df_qsar["Agonist_Activity"].str.lower() == "active") &
    (df_qsar["Agonist_Efficacy (%)"] >= 80) &
    (df_qsar["Viability_Activity"].str.lower() == "inactive")
    )

    # --- HIGH CONFIDENCE INACTIVE ---
    mask_inactive = (
    (df_qsar["Agonist_Activity"].str.lower() == "inactive") &
    (df_qsar["Viability_Activity"].str.lower() == "inactive")
    )

    # Keep only high confidence
    df_qsar = df_qsar[mask_active | mask_inactive].reset_index(drop=True)

    add_report(1, "Active / inactive filtering", before, len(df_qsar))

    print("Actives:", (df_qsar["Agonist_Activity"] == "active").sum())
    print("Inactives:", (df_qsar["Agonist_Activity"] == "inactive").sum())

# =============================================================================
# STEP 2 – SMILES PARSING
# =============================================================================
before = len(df_qsar)

# Drop rows where SMILES is NaN
df_qsar.dropna(subset=[SMILES_COL], inplace=True)
df_qsar = df_qsar.reset_index(drop=True) # Reset index after dropping rows
# Map SMILES to RDKit molecules.
df_qsar["mol"] = df_qsar[SMILES_COL].map(Chem.MolFromSmiles)
# Filter valid rows
df_qsar = df_qsar[df_qsar["mol"].notna()].reset_index(drop=True)

add_report(2, "SMILES parsing", before, len(df_qsar))

# =============================================================================
# STEP 3 – ORGANIC FILTER
# =============================================================================
before = len(df_qsar)

def is_organic(mol):
    return any(a.GetAtomicNum() == 6 for a in mol.GetAtoms())

df_qsar = df_qsar[df_qsar["mol"].map(is_organic)].reset_index(drop=True)

add_report(3, "Filtering non-organic molecules", before, len(df_qsar))

# =============================================================================
# STEP 4 – STANDARDIZATION
# =============================================================================
mols = df_qsar["mol"]

def smiles(m):
    return Chem.MolToSmiles(m, canonical=True)

def report_changes(tag, before, after):
    chg = (before.map(smiles) != after.map(smiles))
    std_report.append({
        "Substep": tag,
        "Molecules": len(before),
        "Changed": chg.sum(),
        "Changed_%": round(chg.mean() * 100, 2)
    })

# 4.1 Normalize
m_norm = mols.map(rdMolStandardize.Normalize)
report_changes("Normalize", mols, m_norm)

# 4.2 Fragment parent (salt stripping)
m_frag = m_norm.map(rdMolStandardize.FragmentParent)
report_changes("FragmentParent", m_norm, m_frag)

# 4.3 Uncharge
uncharger = rdMolStandardize.Uncharger()
m_unch = m_frag.map(uncharger.uncharge)
report_changes("Uncharger", m_frag, m_unch)

# 4.4 Canonical tautomer
te = rdMolStandardize.TautomerEnumerator()
m_std = m_unch.map(te.Canonicalize)
report_changes("TautomerCanonical", m_unch, m_std)

df_qsar["mol_std"] = m_std

add_report(
    4,
    "Structural standardization",
    len(df_qsar),
    len(df_qsar)
)

# =============================================================================
# STEP 5 – CANONICAL SMILES (FROM STANDARDIZED MOL)
# =============================================================================
before = len(df_qsar)

df_qsar[SMILES_COL] = df_qsar["mol_std"].map(
    lambda m: Chem.MolToSmiles(m, canonical=True)
)

df_qsar = df_qsar[~df_qsar[SMILES_COL].str.contains(r"\*", na=False)]
add_report(5, "Canonical SMILES generation", before, len(df_qsar))

# =============================================================================
# STEP 6 – DEDUPLICATION (FINAL)
# =============================================================================
before = len(df_qsar)

df_qsar = df_qsar.drop_duplicates(subset=[SMILES_COL]).reset_index(drop=True)

add_report(6, "Deduplication by standardized SMILES", before, len(df_qsar))

# =============================================================================
# FINALIZATION AND SUMMARY
# =============================================================================
df_qsar.drop(columns=["mol", "mol_std"], inplace=True)

add_report(7, "Summary", total_initial, len(df_qsar))

curation_report_df = pd.DataFrame(curation_report)
std_report_df = pd.DataFrame(std_report)



Actives: 275
Inactives: 7236


In [24]:

# =============================================================================
# SAVE CSV
# =============================================================================
df_qsar.to_csv(f'{BASE_DIR}/ligands/{target}/{target}_ligands_{confidence}.csv', index=False)
curation_report_df.to_csv(f'{BASE_DIR}/ligands/{target}/{target}_curation_report_{confidence}.csv', index=False)
std_report_df.to_csv(f'{BASE_DIR}/ligands/{target}/{target}_standarization_report_{confidence}.csv', index=False)

In [25]:
curation_report_df

,Step,Description,Entries_before,Entries_after,Removed,Removed_%
0,1,Active / inactive filtering,10486,7511,2975,28.37
1,2,SMILES parsing,7511,7385,126,1.68
2,3,Filtering non-organic molecules,7385,7285,100,1.35
3,4,Structural standardization,7285,7285,0,0.00
4,5,Canonical SMILES generation,7285,7285,0,0.00
5,6,Deduplication by standardized SMILES,7285,5625,1660,22.79
6,7,Summary,10486,5625,4861,46.36


In [26]:
std_report_df

,Substep,Molecules,Changed,Changed_%
0,Normalize,7285,12,0.16
1,FragmentParent,7285,1250,17.16
2,Uncharger,7285,404,5.55
3,TautomerCanonical,7285,856,11.75


In [27]:
df_qsar

,CID,SMILES,Agonist_Activity,Agonist_Potency (uM),Agonist_Efficacy (%),Viability_Activity,Viability_Potency (uM),Viability_Efficacy (%),SMILES_raw
0,12850184.0,O=C(O)C(=O)C(O)C(O)C(O)CO,inactive,NaN,0.000,inactive,NaN,0.0,C(C(=O)[C@H]([C@@H]([C@H](C(=O)[O-])O)O)O)O.C(...
1,89753.0,O=C(O)C(O)C(O)C(O)C(O)CO,inactive,NaN,0.000,inactive,NaN,0.0,C([C@H]([C@H]([C@@H]([C@H](C(=O)[O-])O)O)O)O)O...
2,9403.0,C[C@]12CC[C@@H]3c4ccc(O)cc4CC[C@H]3[C@@H]1CC[C...,inactive,NaN,0.000,inactive,NaN,0.0,C[C@]12CC[C@H]3[C@H]([C@@H]1CC[C@@H]2OC(=O)CCC...
3,13218779.0,CC1(C)[C@@H]2CC[C@@]1(C)[C@H](OC(=O)CSC#N)C2,inactive,NaN,0.000,inactive,NaN,0.0,C[C@@]12CC[C@@H](C1(C)C)C[C@H]2OC(=O)CSC#N
4,16043.0,CC(C)(C)c1cc(O)ccc1O,active,16.7855,153.398,inactive,NaN,0.0,CC(C)(C)C1=C(C=CC(=C1)O)O
...,...,...,...,...,...,...,...,...,...
5620,11442.0,C=CCC1=C(C)C(OC(=O)C2C(C=C(C)C)C2(C)C)CC1=O,inactive,NaN,0.000,inactive,NaN,0.0,CC1=C(C(=O)CC1OC(=O)C2C(C2(C)C)C=C(C)C)CC=C
5621,4115.0,COc1ccc(C(c2ccc(OC)cc2)C(Cl)(Cl)Cl)cc1,inactive,NaN,0.000,inactive,NaN,0.0,COC1=CC=C(C=C1)C(C2=CC=C(C=C2)OC)C(Cl)(Cl)Cl
5622,20393.0,CCCCC(CC)COC(=O)c1ccccc1C(=O)O,inactive,NaN,0.000,inactive,NaN,0.0,CCCCC(CC)COC(=O)C1=CC=CC=C1C(=O)O
5623,3289.0,CCCSP(=O)(OCC)SCCC,inactive,NaN,0.000,inactive,NaN,0.0,CCCSP(=O)(OCC)SCCC
